# Лабораторная работа 2. Word2vec, PMI и геометрия эмбеддингов (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 1:** языковое моделирование и представления слов

**Цель.** Реализовать skip-gram with negative sampling (SGNS) с ручным выводом градиентов и экспериментально проверить, что SGNS неявно факторизует сдвинутую матрицу PMI (Levy & Goldberg, 2014).

**Что нужно сдать:**
1. SGNS с ручными градиентами, прошедший проверку градиентов (autograd и конечные разности).
2. Эмбеддинги SVD(PPMI).
3. Сравнение двух моделей на SimLex-999, MEN и аналогиях.
4. Численную проверку теоремы Леви–Голдберга.
5. Анализ анизотропии пространства и эффекта центрирования.

**Теория.** Для пары «центральное слово \(w\) — контекст \(c\)» с \(k\) негативными примерами \(c_N \sim P_n\):
\[
\ell(w, c) = -\log\sigma(\mathbf{w}^\top \mathbf{c}) - \sum_{n=1}^{k}\log\sigma(-\mathbf{w}^\top \mathbf{c}_{N_n}).
\]
Если размерность эмбеддингов достаточна, чтобы каждое произведение \(x = \mathbf{w}^\top\mathbf{c}\) оптимизировалось независимо, оптимум ожидаемой функции потерь достигается при
\[
\mathbf{w}^\top \mathbf{c} = \log\frac{\#(w,c)}{\#(w)\, P_n(c)} - \log k ,
\]
то есть при сдвинутой (на \(\log k\)) PMI относительно распределения негативов.

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

## 0. Корпус text8 и словарь

In [ ]:
import zipfile
from collections import Counter

zpath = download("http://mattmahoney.net/dc/text8.zip", "text8.zip")
with zipfile.ZipFile(zpath) as z:
    raw = z.read("text8").decode("utf-8")
N_CHARS = 10_000_000 if FAST_DEV_RUN else 100_000_000
tokens = raw[:N_CHARS].split()
MIN_COUNT, MAX_VOCAB = 5, (10_000 if FAST_DEV_RUN else 50_000)
freq = Counter(tokens)
itos = [w for w, c in freq.most_common(MAX_VOCAB) if c >= MIN_COUNT]
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)
corpus = np.array([stoi[t] for t in tokens if t in stoi], dtype=np.int64)
counts = np.bincount(corpus, minlength=V).astype(np.float64)
print(f"токенов: {len(corpus):,}, словарь: {V:,}")

## 1. Обучающие пары и распределение негативов

- Окно фиксированной ширины `WINDOW` (без динамического сужения, чтобы проверка теоремы была точной).
- Распределение негативов \(P_n(c) \propto \#(c)^{0.75}\).
- Subsampling частых слов можно включить для основного эксперимента, но для проверки теоремы его нужно отключить: теорема говорит о тех парах, на которых реально идёт обучение.

In [ ]:
WINDOW, K_NEG, DIM = 4, 5, 100

def make_pairs(corpus, window):
    """Все пары (центр, контекст) на расстоянии 1..window в обе стороны."""
    centers, contexts = [], []
    # Для каждого расстояния добавляем оба направления: центр→контекст и обратно.
    for offset in range(1, min(window + 1, len(corpus))):
        left, right = corpus[:-offset], corpus[offset:]
        centers.extend([left, right])
        contexts.extend([right, left])
    return np.concatenate(centers).astype(np.int64), np.concatenate(contexts).astype(np.int64)

centers, contexts = make_pairs(corpus, WINDOW)
neg_probs = counts ** 0.75; neg_probs /= neg_probs.sum()
print(f"пар: {len(centers):,}")

## 2. Градиенты SGNS

Обозначим \(s_+ = \sigma(\mathbf{w}^\top\mathbf{c}_+)\), \(s_n = \sigma(\mathbf{w}^\top\mathbf{c}_{N_n})\). Выведите и реализуйте:
\[
\frac{\partial \ell}{\partial \mathbf{w}} = (s_+ - 1)\,\mathbf{c}_+ + \sum_n s_n \mathbf{c}_{N_n},\qquad
\frac{\partial \ell}{\partial \mathbf{c}_+} = (s_+ - 1)\,\mathbf{w},\qquad
\frac{\partial \ell}{\partial \mathbf{c}_{N_n}} = s_n\,\mathbf{w}.
\]
Функция работает с батчем и **не использует autograd**.

In [ ]:
def sgns_loss_and_grads(Wb, Cpos, Cneg):
    """Wb: (B, d) — векторы центров, Cpos: (B, d) — позитивные контексты, Cneg: (B, k, d) — негативы.
    Возвращает средний по батчу loss и градиенты (gW, gCpos, gCneg) суммарного (не среднего) loss."""
    # d softplus(x) / dx = sigmoid(x), поэтому для позитивной пары
    # коэффициент равен sigmoid(w·c)-1, для каждого негатива — sigmoid(w·c_n).
    with torch.no_grad():
        pos = (Wb * Cpos).sum(-1)
        # Получаем по k скалярных произведений для каждого из B центров.
        neg = torch.einsum('bd,bkd->bk', Wb, Cneg)
        loss = (F.softplus(-pos) + F.softplus(neg).sum(-1)).mean()
        dp, dn = torch.sigmoid(pos) - 1, torch.sigmoid(neg)
        gW = dp[:, None] * Cpos + torch.einsum('bk,bkd->bd', dn, Cneg)
        gCpos = dp[:, None] * Wb
        gCneg = dn[:, :, None] * Wb[:, None, :]
    # loss усреднён по батчу, но gW, gCpos и gCneg относятся к сумме loss.
    # Это важно и для проверки конечными разностями, и для обновления весов.
    return loss, gW, gCpos, gCneg

### Проверка градиентов

1. Сравните с autograd на случайном батче (в `float64`).
2. Выполните проверку конечными разностями по нескольким случайным координатам: \(\frac{\ell(\theta + \varepsilon e_i) - \ell(\theta - \varepsilon e_i)}{2\varepsilon}\).

In [ ]:
torch.manual_seed(0)
B, d, k = 7, 5, 3
Wb = torch.randn(B, d, dtype=torch.float64, requires_grad=True)
Cp = torch.randn(B, d, dtype=torch.float64, requires_grad=True)
Cn = torch.randn(B, k, d, dtype=torch.float64, requires_grad=True)
loss, gW, gCp, gCn = sgns_loss_and_grads(Wb.detach(), Cp.detach(), Cn.detach())
ref = -(F.logsigmoid((Wb * Cp).sum(-1)) + F.logsigmoid(-torch.einsum("bd,bkd->bk", Wb, Cn)).sum(-1)).sum()
ref.backward()
for name, a, b in [("W", gW, Wb.grad), ("Cpos", gCp, Cp.grad), ("Cneg", gCn, Cn.grad)]:
    print(name, "max |diff| =", (a - b).abs().max().item())
    assert torch.allclose(a, b, atol=1e-10)

eps, errors = 1e-6, []
coordinates = np.random.default_rng(SEED).choice(Wb.numel(), 10, replace=False)
for flat_index in coordinates:
    plus, minus = Wb.detach().clone(), Wb.detach().clone()
    plus.view(-1)[flat_index] += eps
    minus.view(-1)[flat_index] -= eps
    lp = sgns_loss_and_grads(plus, Cp.detach(), Cn.detach())[0]
    lm = sgns_loss_and_grads(minus, Cp.detach(), Cn.detach())[0]
    # Функция возвращает средний loss, а градиенты — для суммы по батчу.
    numerical = (lp - lm) * B / (2 * eps)
    errors.append(abs(float(numerical - gW.view(-1)[flat_index])))
print('Конечные разности: max |diff| =', max(errors))
assert max(errors) < 1e-7

## 3. Обучение SGNS

Используйте разреженное обновление SGD: в батче обновляются только строки, встретившиеся в нём (`index_add_`). Learning rate линейно убывает до нуля, как в оригинальном word2vec.

In [ ]:
def train_sgns(centers, contexts, V, dim=DIM, k=K_NEG, epochs=1, batch=512, lr0=0.025, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    W = (torch.rand(V, dim, generator=g) - 0.5) / dim
    C = torch.zeros(V, dim)
    neg_t = torch.tensor(neg_probs, dtype=torch.float)
    n = len(centers); steps = epochs * (n // batch); step = 0; hist = []
    cen_t, ctx_t = torch.from_numpy(centers), torch.from_numpy(contexts)
    for ep in range(epochs):
        perm = torch.randperm(n, generator=g)
        for b in range(n // batch):
            idx = perm[b * batch:(b + 1) * batch]
            wi, ci = cen_t[idx], ctx_t[idx]
            ni = torch.multinomial(neg_t, batch * k, replacement=True, generator=g).view(batch, k)
            lr = lr0 * max(1e-4, 1 - step / steps)
            loss, gW, gCp, gCn = sgns_loss_and_grads(W[wi], C[ci], C[ni])
            # Все градиенты вычислены до изменения весов; повторные индексы
            # суммируются, включая совпадения позитивных и негативных контекстов.
            # index_add_ складывает вклады повторных строк; обычное присваивание
            # по списку индексов не обеспечивает такого накопления.
            W.index_add_(0, wi, gW, alpha=-lr)
            C.index_add_(0, ci, gCp, alpha=-lr)
            C.index_add_(0, ni.reshape(-1), gCn.reshape(-1, dim), alpha=-lr)
            if step % 200 == 0: hist.append((step, loss.item()))
            step += 1
    return W, C, hist

t0 = time.time()
W_sgns, C_sgns, hist = train_sgns(centers, contexts, V, epochs=2 if FAST_DEV_RUN else 5)
print(f"обучение: {time.time() - t0:.0f} c")
plt.plot(*zip(*hist)); plt.xlabel("шаг"); plt.ylabel("loss"); plt.title("SGNS"); plt.show()

## 4. PPMI и SVD

1. Постройте разреженную матрицу совместной встречаемости \(\#(w,c)\) по тем же парам.
2. Вычислите PPMI со сглаживанием распределения контекстов (\(\alpha = 0.75\)):
\(\mathrm{PPMI}(w,c) = \max\big(0, \log\frac{\#(w,c)\,\sum_{c'}\#(c')^{\alpha}}{\#(w)\,\#(c)^{\alpha}}\big)\).
3. Усечённое SVD: \(\mathrm{PPMI} \approx U_d \Sigma_d V_d^\top\), эмбеддинги \(U_d \Sigma_d^{1/2}\).

In [ ]:
import scipy.sparse as sp
from scipy.sparse.linalg import svds

cooc = sp.coo_matrix((np.ones(len(centers), dtype=np.float64), (centers, contexts)), shape=(V, V)).tocsr()
cooc.sum_duplicates()

def ppmi_matrix(cooc, alpha=0.75):
    x = cooc.tocsr().astype(np.float64)
    # Сумма строки — число обучающих пар с данным центром.
    row_counts = np.asarray(x.sum(axis=1)).ravel()
    context_counts = np.asarray(x.sum(axis=0)).ravel() ** alpha
    noise = context_counts / context_counts.sum()
    # PMI считаем только для наблюдавшихся пар. Остальные элементы PPMI остаются нулями.
    entries = x.tocoo()
    values = np.maximum(0.0, np.log(entries.data /
                                  (row_counts[entries.row] * noise[entries.col])))
    result = sp.csr_matrix((values, (entries.row, entries.col)), shape=x.shape)
    result.eliminate_zeros()
    # PPMI обнуляет отрицательные ассоциации и остаётся разреженной.
    # SGNS не отсекает отрицательные произведения явно: это разные цели.
    # Степень .75 повышает относительный вес редких контекстов в шуме.
    return result

t0 = time.time()
P = ppmi_matrix(cooc)
U, S, Vt = svds(P, k=DIM)
W_svd = torch.tensor(U * np.sqrt(S), dtype=torch.float)
print(f"PPMI nnz={P.nnz:,}, SVD: {time.time() - t0:.0f} c")

## 5. Оценка: SimLex-999, MEN, аналогии

- Сходство: ранговая корреляция Спирмена между косинусом и оценками людей (только пары, покрытые словарём; укажите покрытие).
- Аналогии: метод 3CosAdd, \(\arg\max_{x}\cos(x, b - a + a^*)\), исключая слова запроса. Для каждой модели посчитайте точность по семантическим и синтаксическим разделам.

In [ ]:
import re
import pandas as pd
from scipy.stats import spearmanr

base = "https://raw.githubusercontent.com/vecto-ai/word-benchmarks/master/word-similarity/monolingual/en/"
sim_sets = {name: pd.read_csv(download(base + f"{name}.csv", f"{name}.csv")) for name in ("simlex999", "men")}
analogy_lines = download("https://raw.githubusercontent.com/tmikolov/word2vec/master/questions-words.txt",
                         "questions-words.txt").read_text().splitlines()

def normed(E): return F.normalize(E, dim=1)

def eval_similarity(E, df):
    En = normed(E)
    gold, predicted = [], []
    for row in df.itertuples():
        # В MEN к словам приписаны POS-суффиксы, которых нет в text8.
        a = re.sub(r'-(n|v|j)$', '', str(row.word1).lower())
        b = re.sub(r'-(n|v|j)$', '', str(row.word2).lower())
        if a in stoi and b in stoi:
            gold.append(float(row.similarity))
            predicted.append(float(En[stoi[a]] @ En[stoi[b]]))
    rho = float(spearmanr(gold, predicted).statistic) if len(gold) >= 3 else float('nan')
    # SimLex оценивает сходство, MEN — более широкую связанность.
    # Совместные контексты не гарантируют синонимию, поэтому SimLex сложнее.
    return rho, len(gold) / max(1, len(df))

def eval_analogies(E, max_q=None):
    En = normed(E); section = None; res = {"semantic": [0, 0], "syntactic": [0, 0]}
    qs = []
    for line in analogy_lines:
        if line.startswith(":"):
            section = "syntactic" if line.startswith(": gram") else "semantic"; continue
        a, b, a2, b2 = line.lower().split()
        if all(w in stoi for w in (a, b, a2, b2)): qs.append((section, stoi[a], stoi[b], stoi[a2], stoi[b2]))
    if max_q: random.Random(0).shuffle(qs); qs = qs[:max_q]
    for start in range(0, len(qs), 128):
        part = qs[start:start + 128]
        ids = torch.tensor([q[1:] for q in part])
        a, b, a2, expected = ids.T
        query = F.normalize(En[b] - En[a] + En[a2], dim=1)
        scores = query @ En.T
        # Убираем три слова запроса, иначе ответом может оказаться одно из них.
        scores[torch.arange(len(part))[:, None], ids[:, :3]] = -math.inf
        predicted = scores.argmax(1)
        for i, q in enumerate(part):
            res[q[0]][0] += int(predicted[i] == expected[i])
            res[q[0]][1] += 1
    # Доля покрытых словарём вопросов считается до сокращения max_q.
    for group in res:
        all_q, covered_q, current = 0, 0, None
        for line in analogy_lines:
            if line.startswith(':'):
                current = 'syntactic' if line.startswith(': gram') else 'semantic'
            elif current == group:
                all_q += 1
                covered_q += int(all(word in stoi for word in line.lower().split()))
        print(group, 'покрытие:', f'{covered_q}/{all_q}', 'оценено:', res[group][1])
    return {k: (v[0] / max(v[1], 1), v[1]) for k, v in res.items()}

rows = []
for name, E in [("SGNS (W)", W_sgns), ("SGNS (W+C)", W_sgns + C_sgns), ("SVD(PPMI)", W_svd)]:
    r = {"модель": name}
    for s, df in sim_sets.items():
        rho, cov = eval_similarity(E, df); r[s] = round(rho, 3); r[s + " покрытие"] = round(cov, 2)
    an = eval_analogies(E, max_q=3000 if FAST_DEV_RUN else None)
    r["analogy sem"] = round(an["semantic"][0], 3); r["analogy syn"] = round(an["syntactic"][0], 3)
    rows.append(r)
pd.DataFrame(rows)

# В коротком прогоне покрыто около 61% пар SimLex и 42% MEN.
# Корреляции считаются только для пар, где известны оба слова.
# Для аналогий покрытие относится ко всему набору, а точность — к оценённой
# выборке после ограничения max_q. Эти доли отвечают на разные вопросы.
# SVD(PPMI) дал более высокий результат на MEN и аналогиях, SGNS (W+C) — на SimLex.
# Это результаты двух эпох на ограниченном корпусе; порядок моделей может измениться.


## 6. Проверка теоремы Леви–Голдберга

Для частых пар \((w, c)\) (например, \(\#(w,c) \ge 50\)) сравните \(\mathbf{w}^\top\mathbf{c}\) у обученной SGNS с теоретическим оптимумом
\[
\mathrm{SPMI}_k(w,c) = \log\frac{\#(w,c)}{\#(w)\,P_n(c)} - \log k.
\]
Постройте диаграмму рассеяния, посчитайте корреляцию Пирсона и Спирмена, а также среднее отклонение. Объясните, почему совпадение неполное (конечная размерность, недообученность, редкие пары).

In [ ]:
from scipy.stats import pearsonr

entries = cooc.tocoo()
mask = entries.data >= 50
wi, ci, pair_counts = entries.row[mask], entries.col[mask], entries.data[mask]
row_counts = np.asarray(cooc.sum(axis=1)).ravel()
spmi = np.log(pair_counts / (row_counts[wi] * neg_probs[ci])) - np.log(K_NEG)
dots = (W_sgns[wi] * C_sgns[ci]).sum(1).numpy()
print('Частых пар:', len(spmi))
print('Pearson:', pearsonr(spmi, dots).statistic)
print('Spearman:', spearmanr(spmi, dots).statistic)
print('Среднее отклонение:', np.mean(dots - spmi))
print('Среднее абсолютное отклонение:', np.mean(np.abs(dots - spmi)))
plt.figure(figsize=(5, 4))
plt.scatter(spmi, dots, s=4, alpha=.25)
lo, hi = min(spmi.min(), dots.min()), max(spmi.max(), dots.max())
plt.plot([lo, hi], [lo, hi], 'r--')
plt.xlabel('Теоретический оптимум'); plt.ylabel('w·c')
plt.title('SGNS и сдвинутая PMI'); plt.tight_layout(); plt.show()
# n_wc — число вхождений пары, n_w — число пар с данным центром.
# Для отдельного логита x производная ожидаемого loss равна
# n_wc*(sigmoid(x)-1) + k*n_w*P_n(c)*sigmoid(x).
# В нуле производной exp(x)=n_wc/(k*n_w*P_n(c)), откуда формула SPMI.
# Используется реальное P_n(c) из обучения, а не несглаженная PMI.
# Для короткого режима с двумя эпохами расхождение объясняется так:
# Совпадение неполное: логиты связаны общими векторами размерности DIM,
# двух эпох недостаточно для независимого оптимума всех пар; редкие
# пары и семплирование добавляют шум. Последний неполный батч пропущен
# исходным циклом обучения, что также даёт небольшое расхождение.

# В проверенном коротком прогоне Pearson составил 0.935, Spearman — 0.987.
# Значения согласуются с формулой, но не лежат точно на прямой y=x.
# Формула получена для независимо выбираемых логитов; общие векторы
# размерности DIM не позволяют произвольно задать каждое произведение w·c.


## 7. Анизотропия

1. Посчитайте средний косинус между случайными парами слов для каждой модели.
2. Найдите долю дисперсии, объясняемую первыми 5 главными компонентами.
3. Примените «all-but-the-top» (Mu & Viswanath, 2018): вычесть среднее и проекции на первые \(D\) главных компонент. Как меняется качество на SimLex и MEN?

In [ ]:
def anisotropy(E, n=20000):
    En = normed(E); i, j = torch.randint(len(E), (n,)), torch.randint(len(E), (n,))
    return (En[i] * En[j]).sum(1).mean().item()

def all_but_the_top(E, D=3):
    X = E - E.mean(0, keepdim=True)
    if D == 0:
        return X
    _, _, Vh = torch.linalg.svd(X, full_matrices=False)
    # Строки Vh задают направления в пространстве признаков.
    components = Vh[:D]
    # Частотные и общие контексты создают доминирующие направления.
    # Центрирование убирает среднее; проекция удаляет первые D компонент.
    # Вместе с частотным фоном может теряться полезная семантика, поэтому
    # меньший средний косинус не гарантирует улучшения SimLex или MEN.
    return X - (X @ components.T) @ components

for name, E in [("SGNS (W)", W_sgns), ("SVD(PPMI)", W_svd)]:
    X = E - E.mean(0); s = torch.linalg.svdvals(X) ** 2
    print(f"{name:10s} средний cos={anisotropy(E):.3f}  доля 5 ГК={(s[:5].sum() / s.sum()).item():.3f}")
    for D in (0, 1, 3, 7):
        E2 = all_but_the_top(E, D) if D else E - E.mean(0)
        print(f"   D={D}: SimLex={eval_similarity(E2, sim_sets['simlex999'])[0]:.3f}  MEN={eval_similarity(E2, sim_sets['men'])[0]:.3f}  cos={anisotropy(E2):.3f}")

# В коротком прогоне центрирование SGNS (W) уменьшило средний косинус
# с 0.671 до 0.005 и повысило MEN с 0.319 до 0.434. SimLex почти не изменился.
# У SVD(PPMI) изменение MEN было небольшим: с 0.631 до 0.634.
# Удаление ведущих компонент после центрирования снизило SimLex у обеих моделей.
# Поэтому уменьшение среднего косинуса само по себе не доказывает улучшение эмбеддингов.


## Вопросы

1. Выведите оптимум \(\mathbf{w}^\top\mathbf{c}\) для SGNS из условия равенства нулю производной ожидаемой функции потерь по \(x = \mathbf{w}^\top\mathbf{c}\).

   Обозначим частоту пары через $n_{wc}$, а число пар с центром $w$ — через $n_w$. Для одного логита $x$ ожидаемый loss равен $-n_{wc}\log\sigma(x)-k n_wP_n(c)\log\sigma(-x)$. Условие нулевой производной: $n_{wc}(\sigma(x)-1)+k n_wP_n(c)\sigma(x)=0$. Отсюда $e^x=n_{wc}/(k n_wP_n(c))$ и $x=\log[n_{wc}/(n_wP_n(c))]-\log k$. Обычная PMI минус $\log k$ получается, когда $P_n(c)$ совпадает с эмпирическим распределением контекстов.
2. Почему отрицательные значения PMI отбрасывают (PPMI)? Что в SGNS соответствует этому усечению?

   PPMI сохраняет положительные ассоциации и заменяет отрицательные значения нулями. Это даёт разреженную матрицу, с которой удобно работать через SVD. В SGNS скалярные произведения могут быть отрицательными: явного отсечения в нуле нет. Поэтому SVD(PPMI) и SGNS не обязаны давать одинаковые векторы.
3. Почему сглаживание \(P_n(c) \propto \#(c)^{0.75}\) полезно и в SGNS, и в PPMI?

   По сравнению со степенью 1 распределение со степенью 0.75 даёт редким контекстам больший относительный вес и уменьшает долю самых частых. Это меняет набор негативных примеров в SGNS. В формуле PPMI меняется фон, с которым сравнивается частота пары: оценка редкого контекста становится менее завышенной. Степень 0.75 — выбранная эвристика, а не универсальный оптимум.
4. Чем объясняется анизотропия пространства эмбеддингов? Как она связана с частотой слов?

   Одна из причин анизотропии — частые слова и общие контексты, которые создают похожие направления у многих векторов. Поэтому случайные пары могут иметь заметно положительный косинус. Центрирование убирает среднее направление, а all-but-the-top дополнительно удаляет ведущие компоненты. Вместе с частотным фоном может уйти и полезная семантика.
5. Почему SimLex-999 сложнее для дистрибутивных моделей, чем MEN?

   SimLex оценивает сходство, например близость синонимов, а MEN — более широкую связанность. «Кофе» и «чашка» часто встречаются рядом, но не обозначают одно и то же. Модели, обученные на совместной встречаемости, легко улавливают такую связь и могут завысить сходство. Поэтому хорошая корреляция на MEN не гарантирует такую же на SimLex.
